In [ ]:
# Paths: set DATA_DIR / OUTPUT_DIR as environment variables, or edit the defaults below.
import os
DATA_DIR = os.environ.get("DATA_DIR", "../data")


In [ ]:
import pandas as pd
import numpy as np
import joblib
from datetime import datetime
from sklearn.neighbors import BallTree

In [ ]:
pass  # Colab only: from google.colab import drive
pass  # Colab only: drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR     = DATA_DIR + "/"
DATASETS_DIR = BASE_DIR + "datasets/"
MODELS_DIR   = BASE_DIR + "models/"
NATIONAL_DIR = BASE_DIR + "national/"
REGIONS_DIR  = BASE_DIR + "regions/"

model        = joblib.load(MODELS_DIR + 'risk_model_v3_national.pkl')
le_time      = joblib.load(MODELS_DIR + 'le_time_v3.pkl')
le_season    = joblib.load(MODELS_DIR + 'le_season_v3.pkl')
le_surface   = joblib.load(MODELS_DIR + 'le_surface_v3.pkl')
grid_history = pd.read_csv(NATIONAL_DIR + 'national_grid_history.csv')

print(f"✅ Model loaded")
print(f"✅ Grid cells: {len(grid_history):,}")
print(f"   Coverage: lat {grid_history['avg_lat'].min():.2f}→{grid_history['avg_lat'].max():.2f}, "
      f"lon {grid_history['avg_lon'].min():.2f}→{grid_history['avg_lon'].max():.2f}")

✅ Model loaded
✅ Grid cells: 43,903
   Coverage: lat 49.97→58.60, lon -6.58→1.76


In [ ]:
import glob
import os
from sklearn.preprocessing import MinMaxScaler

# Check Manchester coverage before fix
manc_before = grid_history[
    (grid_history['avg_lat'] >= 53.35) & (grid_history['avg_lat'] <= 53.65) &
    (grid_history['avg_lon'] >= -2.75) & (grid_history['avg_lon'] <= -1.90)
]
print(f"Manchester cells before fix: {len(manc_before):,}")

# Copy Manchester pilot file into regions folder if not already there
manc_region_path = REGIONS_DIR + "region_10_Greater_Manchester.csv"
if not os.path.exists(manc_region_path):
    manc_pilot = pd.read_csv(DATASETS_DIR + 'grid_history.csv')
    manc_pilot['region_id']   = '10'
    manc_pilot['region_name'] = 'Greater_Manchester'
    manc_pilot.to_csv(manc_region_path, index=False)
    print(f"✅ Manchester pilot saved to regions folder ({len(manc_pilot):,} cells)")
else:
    print(f"✅ Manchester region file already exists")

# Re-merge all region files
region_files = sorted(glob.glob(REGIONS_DIR + "region_*.csv"))
print(f"\nMerging {len(region_files)} region files...")

frames = []
for f in region_files:
    df = pd.read_csv(f)
    frames.append(df)

national_grid_new = pd.concat(frames, ignore_index=True)

# Keep highest accident count when deduplicating border cells
before = len(national_grid_new)
national_grid_new = national_grid_new.sort_values(
    'total_accidents', ascending=False
).drop_duplicates(subset='grid_id', keep='first')
after = len(national_grid_new)
print(f"Duplicates removed: {before - after:,}")
print(f"Final grid cells:   {after:,}")

# Re-normalise nationally
scaler = MinMaxScaler()
national_grid_new['hist_freq_score']     = scaler.fit_transform(national_grid_new[['log_freq']])
national_grid_new['hist_severity_score'] = scaler.fit_transform(national_grid_new[['serious_rate']])
national_grid_new['hist_speed_score']    = scaler.fit_transform(national_grid_new[['avg_speed_limit']])
national_grid_new['base_risk_score']     = (
    0.60 * national_grid_new['hist_freq_score'] +
    0.30 * national_grid_new['hist_severity_score'] +
    0.10 * national_grid_new['hist_speed_score']
)

# Save and reload
national_grid_new.to_csv(NATIONAL_DIR + 'national_grid_history.csv', index=False)
grid_history = national_grid_new.copy()

manc_after = grid_history[
    (grid_history['avg_lat'] >= 53.35) & (grid_history['avg_lat'] <= 53.65) &
    (grid_history['avg_lon'] >= -2.75) & (grid_history['avg_lon'] <= -1.90)
]
print(f"\nManchester cells after fix: {len(manc_after):,}")
print(f" national_grid_history.csv updated")

Manchester cells before fix: 1,821
✅ Manchester region file already exists

Merging 37 region files...
Duplicates removed: 4,331
Final grid cells:   43,903

Manchester cells after fix: 1,822
 national_grid_history.csv updated


In [ ]:
# Build BallTree spatial index for fast nearest-neighbour lookups
grid_coords_rad = np.radians(grid_history[['avg_lat','avg_lon']].values)
grid_tree       = BallTree(grid_coords_rad, metric='haversine')
print(f" Spatial index built on {len(grid_history):,} grid cells")

 Spatial index built on 43,903 grid cells


In [ ]:
def get_time_bucket(hour):
    if 0  <= hour < 6:  return 'late_night'
    if 6  <= hour < 9:  return 'morning_rush'
    if 9  <= hour < 12: return 'mid_morning'
    if 12 <= hour < 14: return 'lunch'
    if 14 <= hour < 17: return 'afternoon'
    if 17 <= hour < 20: return 'evening_rush'
    return 'night'

def get_season(month):
    if month in [12, 1, 2]: return 'winter'
    if month in [3, 4, 5]:  return 'spring'
    if month in [6, 7, 8]:  return 'summer'
    return 'autumn'

def get_segment_history(lat, lon, max_radius_km=15.0):
    """
    Find nearest grid cell within max_radius_km.
    Falls back to national medians if nothing found within radius.
    """
    point_rad          = np.radians([[lat, lon]])
    distances, indices = grid_tree.query(point_rad, k=1)
    dist_km            = distances[0][0] * 6371.0
    idx                = indices[0][0]

    if dist_km > max_radius_km:
        return {
            'base_risk_score':     float(grid_history['base_risk_score'].median()),
            'log_freq':            float(grid_history['log_freq'].median()),
            'serious_rate':        float(grid_history['serious_rate'].median()),
            'hist_freq_score':     float(grid_history['hist_freq_score'].median()),
            'hist_severity_score': float(grid_history['hist_severity_score'].median()),
            'pct_dark':            float(grid_history['pct_dark'].median()),
            'pct_wet':             float(grid_history['pct_wet'].median()),
            'pct_rural':           float(grid_history['pct_rural'].median()),
            'pct_high_speed':      float(grid_history['pct_high_speed'].median()),
            '_nearest_grid_km':    round(dist_km, 3),
            '_fallback':           True,
        }

    nearest = grid_history.iloc[idx]
    return {
        'base_risk_score':     float(nearest['base_risk_score']),
        'log_freq':            float(nearest['log_freq']),
        'serious_rate':        float(nearest['serious_rate']),
        'hist_freq_score':     float(nearest['hist_freq_score']),
        'hist_severity_score': float(nearest['hist_severity_score']),
        'pct_dark':            float(nearest['pct_dark']),
        'pct_wet':             float(nearest['pct_wet']),
        'pct_rural':           float(nearest['pct_rural']),
        'pct_high_speed':      float(nearest['pct_high_speed']),
        '_nearest_grid_km':    round(dist_km, 3),
        '_fallback':           False,
    }

In [ ]:
def score_location(lat, lon, dt, weather_code, surface_code,
                   speed_limit, road_type, urban_rural,
                   is_junction=0, junction_detail=0):
    """
    Score a road location given current conditions.

    weather_code:  1=Fine 2=Raining 3=Snowing 5=Rain+Wind 7=Fog
    surface_code:  1=Dry 2=Wet 3=Snow 4=Ice
    urban_rural:   1=Urban 2=Rural
    road_type:     1=Roundabout 3=Dual 6=Single 7=Slip
    """
    hour  = dt.hour
    month = dt.month
    dow   = dt.weekday()

    time_bucket = get_time_bucket(hour)
    season      = get_season(month)

    is_dark           = 1 if hour < 7 or hour >= 20 else 0
    is_dark_no_lights = 1 if (is_dark and urban_rural == 2) else 0
    is_wet_road       = 1 if surface_code == 2 else 0
    is_icy_road       = 1 if surface_code in [3, 4] else 0
    is_bad_weather    = 1 if weather_code in [2, 3, 5, 6, 7] else 0
    is_high_speed     = 1 if speed_limit >= 60 else 0
    is_rural_road     = 1 if urban_rural == 2 else 0
    is_dual           = 1 if road_type == 3 else 0
    is_weekend        = 1 if dow >= 5 else 0
    is_rush_hour      = 1 if hour in [7,8,9,16,17,18] else 0
    friday_evening    = 1 if (dow == 4 and time_bucket == 'evening_rush') else 0

    dark_and_wet        = is_dark * is_wet_road
    high_speed_and_dark = is_high_speed * is_dark
    high_speed_and_wet  = is_high_speed * is_wet_road
    rural_and_dark      = is_rural_road * is_dark
    dark_and_no_lights  = is_dark_no_lights

    try:
        time_enc = le_time.transform([time_bucket])[0]
    except:
        time_enc = 0
    try:
        season_enc = le_season.transform([season])[0]
    except:
        season_enc = 0

    surface_label_map = {1:'Dry', 2:'Wet/Damp', 3:'Snow', 4:'Frost/Ice'}
    surface_label = surface_label_map.get(surface_code, 'Unknown')
    try:
        surface_enc = le_surface.transform([surface_label])[0]
    except:
        surface_enc = 0

    hist = get_segment_history(lat, lon)

    features = pd.DataFrame([{
        'hour':                hour,
        'day_of_week':         dow,
        'month':               month,
        'time_bucket_enc':     time_enc,
        'season_enc':          season_enc,
        'is_weekend':          is_weekend,
        'is_rush_hour':        is_rush_hour,
        'friday_evening':      friday_evening,
        'is_dark':             is_dark,
        'is_dark_no_lights':   is_dark_no_lights,
        'light_conditions':    6 if is_dark_no_lights else (4 if is_dark else 1),
        'is_wet_road':         is_wet_road,
        'is_icy_road':         is_icy_road,
        'is_bad_weather':      is_bad_weather,
        'surface_enc':         surface_enc,
        'weather_conditions':  weather_code,
        'speed_limit':         speed_limit,
        'road_type':           road_type,
        'urban_or_rural_area': urban_rural,
        'is_dual_carriageway': is_dual,
        'is_high_speed':       is_high_speed,
        'is_rural_road':       is_rural_road,
        'at_junction':         is_junction,
        'junction_detail':     junction_detail,
        'dark_and_wet':        dark_and_wet,
        'high_speed_and_dark': high_speed_and_dark,
        'high_speed_and_wet':  high_speed_and_wet,
        'rural_and_dark':      rural_and_dark,
        'dark_and_no_lights':  dark_and_no_lights,
        'base_risk_score':     hist['base_risk_score'],
        'log_freq':            hist['log_freq'],
        'serious_rate':        hist['serious_rate'],
        'hist_freq_score':     hist['hist_freq_score'],
        'hist_severity_score': hist['hist_severity_score'],
        'pct_dark':            hist['pct_dark'],
        'pct_wet':             hist['pct_wet'],
        'pct_rural':           hist['pct_rural'],
        'pct_high_speed':      hist['pct_high_speed'],
    }])

    raw_score = model.predict(features)[0]
    return round(float(np.clip(raw_score * 100, 0, 100)), 1)

In [ ]:
def generate_final_score(raw_model_score, conditions, speed_limit):
    """
    Apply condition multipliers on top of raw model score.
    Dangerous conditions elevate, calm conditions dampen.
    """
    score = raw_model_score

    if conditions['is_icy_road']:
        score = max(score, 50)
        score *= 1.25
    if conditions['is_dark_no_lights'] and speed_limit >= 60:
        score = max(score, 45)
        score *= 1.15
    if conditions['is_bad_weather'] and speed_limit >= 60:
        score *= 1.10
    if conditions['friday_evening']:
        score *= 1.05
    if (not conditions['is_dark'] and
        not conditions['is_wet_road'] and
        not conditions['is_bad_weather'] and
        not conditions['is_icy_road'] and
        speed_limit <= 30):
        score *= 0.70

    return round(min(max(score, 0), 100), 1)

In [ ]:
def generate_alert(risk_score, conditions):
    """Turn a risk score into a tiered driver alert with specific advice."""

    if risk_score >= 65:
        tier, color = 'CRITICAL', '🔴'
    elif risk_score >= 45:
        tier, color = 'HIGH', '🟠'
    elif risk_score >= 25:
        tier, color = 'MEDIUM', '🟡'
    else:
        tier, color = 'LOW', '🟢'

    advice = []
    if conditions['is_dark_no_lights']:
        advice.append("Unlit road ahead — use full beam where safe")
    elif conditions['is_dark']:
        advice.append("Reduced visibility — increase following distance")
    if conditions['is_icy_road']:
        advice.append("Ice on road — reduce speed significantly, brake gently")
    elif conditions['is_wet_road']:
        advice.append("Wet road surface — allow extra stopping distance")
    if conditions['is_bad_weather']:
        advice.append("Poor weather — reduce speed and increase alertness")
    if conditions['is_high_speed'] and conditions['is_dark']:
        advice.append("High speed limit in dark conditions — drive to your headlights")
    if conditions['friday_evening']:
        advice.append("Friday evening — statistically highest risk period of the week")
    if not advice:
        advice.append("Stay alert and maintain safe following distance")

    return {'risk_score': risk_score, 'tier': tier, 'color': color, 'advice': advice}

In [ ]:
print(" Nearest-neighbour lookup test ")
test_coords = [
    ('Friday hotspot',     53.480, -2.240),
    ('Rural A-road night', 53.55,  -2.50),
    ('Motorway midday',    53.42,  -2.35),
    ('Icy rural',          53.58,  -2.45),
    ('Sunday urban',       53.48,  -2.23),
]

for name, lat, lon in test_coords:
    h = get_segment_history(lat, lon)
    fallback = '* FALLBACK' if h.get('_fallback') else '**'
    print(f"  {fallback} {name:25s} → base_risk={h['base_risk_score']:.3f}  "
          f"log_freq={h['log_freq']:.3f}  nearest={h['_nearest_grid_km']}km")

 Nearest-neighbour lookup test 
  ** Friday hotspot            → base_risk=0.573  log_freq=3.266  nearest=0.036km
  ** Rural A-road night        → base_risk=0.289  log_freq=1.482  nearest=0.186km
  ** Motorway midday           → base_risk=0.195  log_freq=0.788  nearest=0.206km
  ** Icy rural                 → base_risk=0.392  log_freq=2.001  nearest=0.18km
  ** Sunday urban              → base_risk=0.466  log_freq=2.667  nearest=0.097km


In [ ]:
scenarios = [
    {
        'name':        'Friday evening, wet, known hotspot',
        'lat':          53.480, 'lon': -2.240,
        'dt':           datetime(2024, 11, 15, 17, 30),
        'weather_code': 2, 'surface_code': 2,
        'speed_limit':  30, 'road_type': 6,
        'urban_rural':  1, 'is_junction': 1,
    },
    {
        'name':        'Rural A-road, night, no lighting, dry',
        'lat':          53.55, 'lon': -2.50,
        'dt':           datetime(2024, 11, 14, 23, 0),
        'weather_code': 1, 'surface_code': 1,
        'speed_limit':  60, 'road_type': 6,
        'urban_rural':  2, 'is_junction': 0,
    },
    {
        'name':        'Motorway, midday, fine weather',
        'lat':          53.42, 'lon': -2.35,
        'dt':           datetime(2024, 11, 13, 12, 0),
        'weather_code': 1, 'surface_code': 1,
        'speed_limit':  70, 'road_type': 3,
        'urban_rural':  2, 'is_junction': 0,
    },
    {
        'name':        'Icy rural road, early morning',
        'lat':          53.58, 'lon': -2.45,
        'dt':           datetime(2024, 1, 10, 7, 15),
        'weather_code': 3, 'surface_code': 4,
        'speed_limit':  60, 'road_type': 6,
        'urban_rural':  2, 'is_junction': 0,
    },
    {
        'name':        'Sunday afternoon, dry, urban 20mph zone',
        'lat':          53.48, 'lon': -2.23,
        'dt':           datetime(2024, 6, 9, 14, 0),
        'weather_code': 1, 'surface_code': 1,
        'speed_limit':  20, 'road_type': 6,
        'urban_rural':  1, 'is_junction': 0,
    },
]

for s in scenarios:
    hour = s['dt'].hour
    conditions = {
        'is_dark':           1 if hour < 7 or hour >= 20 else 0,
        'is_dark_no_lights': 1 if ((hour < 7 or hour >= 20) and s['urban_rural'] == 2) else 0,
        'is_wet_road':       1 if s['surface_code'] == 2 else 0,
        'is_icy_road':       1 if s['surface_code'] in [3, 4] else 0,
        'is_bad_weather':    1 if s['weather_code'] in [2, 3, 5, 6, 7] else 0,
        'is_high_speed':     1 if s['speed_limit'] >= 60 else 0,
        'friday_evening':    1 if (s['dt'].weekday() == 4 and 17 <= hour < 20) else 0,
    }

    raw_score   = score_location(
        lat=s['lat'], lon=s['lon'], dt=s['dt'],
        weather_code=s['weather_code'], surface_code=s['surface_code'],
        speed_limit=s['speed_limit'], road_type=s['road_type'],
        urban_rural=s['urban_rural'], is_junction=s.get('is_junction', 0)
    )
    final_score = generate_final_score(raw_score, conditions, s['speed_limit'])
    alert       = generate_alert(final_score, conditions)

    print(f"Scenario: {s['name']}")
    print(f"  Model raw: {raw_score:.1f} → Final: {final_score}/100")
    print(f"  {alert['color']} [{alert['tier']}]")
    for a in alert['advice']:
        print(f"  → {a}")
    print()

Scenario: Friday evening, wet, known hotspot
  Model raw: 53.4 → Final: 56.1/100
  🟠 [HIGH]
  → Wet road surface — allow extra stopping distance
  → Poor weather — reduce speed and increase alertness
  → Friday evening — statistically highest risk period of the week

Scenario: Rural A-road, night, no lighting, dry
  Model raw: 29.2 → Final: 51.7/100
  🟠 [HIGH]
  → Unlit road ahead — use full beam where safe
  → High speed limit in dark conditions — drive to your headlights

Scenario: Motorway, midday, fine weather
  Model raw: 18.0 → Final: 18.0/100
  🟢 [LOW]
  → Stay alert and maintain safe following distance

Scenario: Icy rural road, early morning
  Model raw: 35.8 → Final: 68.8/100
  🔴 [CRITICAL]
  → Ice on road — reduce speed significantly, brake gently
  → Poor weather — reduce speed and increase alertness

Scenario: Sunday afternoon, dry, urban 20mph zone
  Model raw: 44.6 → Final: 31.2/100
  🟡 [MEDIUM]
  → Stay alert and maintain safe following distance

